# Fantasy Basketball AI Ranker — Portfolio Analysis

**Stack:** Python · scikit-learn · Anthropic API · SQLite · Flask

---

This notebook walks through the full three-layer system I built to rank NBA players for ESPN H2H points leagues — the way *I* would rank them, not the way a generic model would.

## 1. Project Overview

### What this is

Most fantasy tools give you the same rankings. This project goes further: it trains a **sklearn logistic regression on my own draft history and head-to-head player comparisons** to learn what I actually value, then produces a top-20 ranking entirely from my trained models — no LLM involved.

### The Scoring System

ESPN H2H Points scoring:

| Stat | Points | Derived value |
|------|--------|---------------|
| PTS  | +1     | — |
| 3PM  | +1     | 3-pointer made = **5 pts** (3+1+2−1) |
| FGM  | +2     | 2-pointer made = **3 pts** (2+2−1) |
| FGA  | −1     | — |
| FTM  | +1     | Free throw made = **1 pt** (1+1−1) |
| FTA  | −1     | — |
| REB  | +1     | — |
| AST  | +2     | — |
| STL  | +4     | — |
| BLK  | +4     | — |
| TOV  | −2     | — |

### Three-Layer Architecture

```
ESPN Fantasy API          nba_api
(player pool,             (per-game stats for
 injury status)            fantasy-relevant players only)
        │                          │
        └──────────┬───────────────┘
                   ▼
            ETL Pipeline (Python)
            SQLite Database
                   │
        ┌──────────┴──────────────────┐
        ▼                             ▼
 Stat Model (sklearn)       Intuition Model (sklearn)
 LinearRegression →         LogisticRegression trained
 fantasy PPG baseline       on MY swipe history + drafts
 4 diagnostic plots         13-feature delta vector
        │                             │
        └──────────┬──────────────────┘
                   ▼
            Pure sklearn Ranker
            Round-robin tournament
            (intuition model picks winner
             of every pair; stat model
             sets the baseline pool)
                   │
                   ▼
            Top 20 Rankings
            saved to outputs/
```

**No LLM anywhere in this pipeline.** The intuition model IS the preference engine — it's a logistic regression I trained on my own decisions.

### Delta vector features (what the model learns from)

| Feature | What it captures |
|---------|-----------------|
| delta_pts, fg3m, ast, stl, blk, reb, tov | Core per-game stat differences |
| delta_fantasy_ppg | ESPN H2H projected PPG difference |
| delta_consistency | Std dev difference (reliability) |
| **delta_is_rookie** | Experience edge — is one player in their first season? |
| **delta_team_win_pct** | Did one player play on a better team last year? |
| **delta_position** | Guard (PG=1) → Big (C=5) — captures position preference |
| **delta_gp_prev** | Prior-season games played — injury risk proxy |

## 2. Data Pipeline

The ETL layer pulls three seasons of per-game stats from the NBA API, converts them to ESPN fantasy points per game, and computes a consistency score (std dev of game-by-game fantasy output) for each player. Everything lands in a SQLite database — lightweight, portable, no setup required.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from db import get_conn

conn = get_conn()

top20 = pd.read_sql_query("""
    SELECT
        ROW_NUMBER() OVER (ORDER BY fs.fantasy_ppg DESC) AS rank,
        p.name, p.team, p.position, ps.gp,
        ROUND(ps.pts, 1)              AS pts,
        ROUND(ps.ast, 1)              AS ast,
        ROUND(ps.stl, 1)              AS stl,
        ROUND(ps.blk, 1)              AS blk,
        ROUND(fs.fantasy_ppg, 2)      AS fantasy_ppg,
        ROUND(fs.consistency_score, 2) AS std_dev
    FROM fantasy_scores fs
    JOIN players p       ON p.player_id = fs.player_id
    JOIN player_stats ps ON ps.player_id = fs.player_id AND ps.season = fs.season
    WHERE fs.season = '2024-25'
      AND ps.gp >= 15
    ORDER BY fs.fantasy_ppg DESC
    LIMIT 20
""", conn)

conn.close()

top20.style \
    .background_gradient(subset=['fantasy_ppg'], cmap='YlOrRd') \
    .background_gradient(subset=['std_dev'], cmap='RdYlGn_r') \
    .format({'fantasy_ppg': '{:.2f}', 'std_dev': '{:.2f}'}) \
    .set_caption('Top 20 Players by Fantasy PPG — 2024-25 Season')

## 3. Scoring Engine

`compute_fantasy_score()` is the core translation layer — raw box score stats in, ESPN H2H fantasy points out. Every downstream model uses this same function, so the math is consistent everywhere.

In [ ]:
from etl.scoring import compute_fantasy_score, SCORING_WEIGHTS

# Five players that illustrate different fantasy archetypes
star_players = [
    {'name': 'Nikola Jokic',            'pts': 26.4, 'fg3m': 0.9, 'fgm': 9.0,  'fga': 15.0, 'ftm': 5.5, 'fta': 6.2,  'reb': 12.3, 'ast': 9.0,  'stl': 1.4, 'blk': 0.9, 'tov': 3.0},
    {'name': 'Shai Gilgeous-Alexander', 'pts': 32.7, 'fg3m': 1.5, 'fgm': 10.8, 'fga': 20.2, 'ftm': 9.1, 'fta': 10.3, 'reb': 5.1,  'ast': 6.4,  'stl': 2.0, 'blk': 0.9, 'tov': 2.7},
    {'name': 'Giannis Antetokounmpo',   'pts': 30.4, 'fg3m': 0.5, 'fgm': 11.2, 'fga': 18.4, 'ftm': 7.3, 'fta': 10.8, 'reb': 11.6, 'ast': 6.5,  'stl': 1.2, 'blk': 1.1, 'tov': 3.4},
    {'name': 'Tyrese Haliburton',       'pts': 21.6, 'fg3m': 3.1, 'fgm': 7.4,  'fga': 15.2, 'ftm': 3.6, 'fta': 4.1,  'reb': 4.4,  'ast': 10.9, 'stl': 1.7, 'blk': 0.4, 'tov': 3.6},
    {'name': 'Alex Caruso',             'pts': 13.1, 'fg3m': 1.8, 'fgm': 4.6,  'fga': 9.2,  'ftm': 2.1, 'fta': 2.4,  'reb': 4.2,  'ast': 4.8,  'stl': 2.4, 'blk': 0.7, 'tov': 1.4},
]

rows = []
for p in star_players:
    name  = p.pop('name')
    total = compute_fantasy_score(**p)
    breakdown = {stat.upper(): round(p[stat] * SCORING_WEIGHTS[stat], 2)
                 for stat in SCORING_WEIGHTS if stat in p}
    rows.append({'Player': name, 'Total Fantasy PPG': round(total, 2), **breakdown})

pd.DataFrame(rows).set_index('Player').style \
    .background_gradient(subset=['Total Fantasy PPG'], cmap='YlOrRd') \
    .format('{:.2f}') \
    .set_caption('Stat → Fantasy Point Contribution by Category')

## 4. Statistical Model Results

Two sklearn linear regressions:
1. **Predictive** — trained on 2022-23 + 2023-24, tested on 2024-25. Shows how well past performance predicts current fantasy value.
2. **Feature importance** — fit on all three seasons. Coefficients quantify which raw stats drive the most fantasy value under this scoring system.

Four diagnostic plots are generated and saved to `outputs/`.

In [ ]:
from models.stat_model import load_data, run_predictive_model, run_feature_importance

df      = load_data()
results = run_predictive_model(df)
coefs   = run_feature_importance(df)

In [ ]:
from IPython.display import Image, display
from pathlib import Path

outputs = Path('../outputs')

for plot_file in ['player_tiers.png', 'stat_correlation_heatmap.png',
                  'positional_value_boxplot.png', 'predicted_vs_actual.png']:
    p = outputs / plot_file
    if p.exists():
        print(f'--- {plot_file} ---')
        display(Image(filename=str(p)))
    else:
        print(f'{plot_file} not found — run: python main.py stat-model')

In [ ]:
# Which raw stats drive fantasy value most under this scoring?
interp = {
    'stl': 'Highest multiplier — elite steal contributors are massively undervalued in standard rankings',
    'blk': 'Same as STL — rim protectors get a huge boost vs. raw-points thinking',
    'ast': 'Double per assist — playmakers score way higher than points-only tells you',
    'reb': 'Linear value — solid but not premium',
    'pts': 'Direct but no multiplier — pure scorers lose ground to efficient ones',
    'fgm': 'Hidden bonus — every make adds 2 on top of PTS',
    'ftm': 'One-for-one — clean FT shooters valued fairly',
    'tov': 'Costs double — high-usage scorers with TOs get crushed',
    'fga': 'Costs a point per attempt — volume shooters with low % hurt themselves',
    'fta': 'Costs a point per attempt — FT-heavy players who miss pay the price',
    'fg3m': 'Slight premium on top of FGM/FGA value',
}
coefs['interpretation'] = coefs['feature'].map(interp)
coefs.style.set_caption('Feature Importance — What Drives Fantasy Value').hide(axis='index')

## 5. Intuition Model

The differentiator. A logistic regression trained on:
- **Head-to-head swipes** — real preference data from the swipe UI
- **Synthetic draft signals** — every time I drafted Player A before Player B, the model treats that as an implicit preference, weighted by round differential

The feature vector is a **delta vector**: `[Δpts, Δfg3m, Δast, Δstl, Δblk, Δreb, Δtov, Δfantasy_ppg, Δconsistency]`. The model learns *which stat differences drive my choices*, not just which players I like.

In [ ]:
from models.intuition_model import get_model_confidence, get_preference_profile

print(f'Model confidence: {get_model_confidence()}\n')
print(get_preference_profile())

In [ ]:
# Feature importance bar chart
import joblib
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')

FEATURES = ['delta_pts', 'delta_fg3m', 'delta_ast', 'delta_stl',
            'delta_blk', 'delta_reb', 'delta_tov', 'delta_fantasy_ppg', 'delta_consistency']

model_files = sorted(Path('../outputs').glob('intuition_model_*.pkl'), reverse=True)
if model_files:
    artifact = joblib.load(model_files[0])
    coef_vals = artifact['clf'].coef_[0]

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')
    colors = ['#4ade80' if c > 0 else '#f87171' for c in coef_vals]
    ax.barh(FEATURES, coef_vals, color=colors, alpha=0.85)
    ax.axvline(0, color='white', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('Coefficient (positive = favored in winner)', color='white')
    ax.set_title('Intuition Model — Feature Importance', color='white', fontsize=13)
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_color('#444')
    plt.tight_layout()
    plt.savefig('../outputs/feature_importance.png', dpi=120, facecolor=fig.get_facecolor())
    plt.show()
else:
    print('No trained model — run: python main.py train')

## 6. Rankings Output

Rankings are produced entirely by two sklearn models — no LLM:

1. **Stat model** (LinearRegression) provides the baseline pool: top 40 players by projected fantasy PPG
2. **Intuition model** (LogisticRegression) runs a round-robin tournament: every player faces every other player, the model predicts who I'd prefer in each matchup based on the 13-feature delta vector
3. Final rank = 65% stat score + 35% intuition score

The 35/65 blend is configurable in `models/ranker.py` (`_BLEND_ALPHA`). Higher alpha = rankings that reflect my taste more than raw production.

**Temporal correctness:** When the intuition model learned from my 2022-23 draft picks, it used 2022-23 stats — including correct rookie flags for that year. A 2022-23 rookie appears with `is_rookie=1` in the 2022-23 training data even if they're a veteran today.

## 6. Rankings Output

Rankings are produced entirely by two sklearn models — no LLM:

1. **Stat model** (LinearRegression) provides the baseline pool: top 40 players by projected fantasy PPG
2. **Intuition model** (LogisticRegression) runs a round-robin tournament: every player faces every other player, the model predicts who I'd prefer in each matchup based on the 13-feature delta vector
3. Final rank = 65% stat score + 35% intuition score

The 35/65 blend is configurable in `models/ranker.py` (`_BLEND_ALPHA`). Higher alpha = rankings that reflect my taste more than raw production.

In [ ]:
from IPython.display import Markdown

ranking_files = sorted(outputs.glob("weekly_rankings_*.md"), reverse=True)
if ranking_files:
    display(Markdown(ranking_files[0].read_text()))
else:
    print("No rankings file — run: python main.py rankings")

In [ ]:
from models.ranker import generate_rankings

# Uncomment to regenerate rankings inline:
# generate_rankings()

## 7. Conclusions

### What the model learned

- **STL and BLK are my primary differentiators.** I consistently chose players with elite defensive stats over pure scorers — which makes sense: both are worth 4× a point under ESPN's scoring.
- **I prioritize AST over REB** in close matchups. Playmakers who also contribute defensively are my sweet spot.
- **I've historically drafted guards early.** The draft history shows a consistent pattern of PG/SG picks in rounds 1-3.
- **I underweight inefficient scorers.** High FGA with poor FGM ratios drag down fantasy PPG more than most people intuit, and my swipe history shows I've internalized this.

### What I'd add next

- **Waiver wire recommender** — surface top available players each week based on my preference profile
- **Trade analyzer** — evaluate proposed trades using stat model + intuition weighting
- **Opponent scouting** — analyze my upcoming opponent's roster weaknesses
- **Injury news scraping** — automate the injury context input so I don't have to type it manually
- **Multi-season preference drift** — track how my preferences change year over year